In [ ]:
import numpy as np
import xarray as xr
from glob import glob
import random

import os
from tqdm.notebook import tqdm
import re

import pop_tools

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import StrMethodFormatter
from matplotlib.image import imread
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec

from mpl_toolkits.axes_grid1.inset_locator import inset_axes


import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.img_tiles as cimgt
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

import util

from scipy.spatial import ConvexHull, Delaunay

In [ ]:
oae_curves = xr.open_dataset('../data/oae_efficiency_curves.nc')
dor_curves = xr.open_dataset('../data/dor_efficiency_curves.nc')
weighted_eta_max = xr.open_dataset('../data/weighted_eta_max_curves.nc')
eta_max = xr.open_dataset('../data/eta_max_mean.nc')

In [ ]:
def plot_polygons(ax, final_polygon_mask_atlantic, final_polygon_vertices_atlantic, cluster_centers_atlantic, ii):

    vertices = np.array(final_polygon_vertices_atlantic[ii])
    # plot convex hull
    if len(vertices) >= 3:
        hull = ConvexHull(vertices) # get the indices of the periphery of a patch, for plotting purpose
        polygon = list(vertices[iii] for iii in hull.vertices.tolist() )
        polygon = np.array(polygon)

        ax.plot(np.append(polygon[:,0], polygon[0,0]), np.append(polygon[:,1], polygon[0,1]), 'k', linewidth=1.5, transform=ccrs.PlateCarree())

    # plot polygon masks
    index = np.where(final_polygon_mask_atlantic[ii] == 1)
    ax.scatter(tlong[index], tlat[index], c='gray', s=1, alpha=1, transform=ccrs.PlateCarree())

def no_nans(arr):
    return arr[~np.isnan(arr)]

In [ ]:
grid_name = 'POP_gx1v7'
grid = pop_tools.get_grid(grid_name)
tlong = grid.TLONG.values
tlat = grid.TLAT.values

# atlantic polygon masks
final_polygon_mask_atlantic = np.load('../data/polygon_data/Atlantic_final_polygon_mask.npy', allow_pickle=True)
final_polygon_vertices_atlantic = np.load('../data/polygon_data/Atlantic_final_polygon_vertices.npy', allow_pickle=True)
cluster_centers_atlantic = np.load('../data/polygon_data/Atlantic_final_cluster_centers.npy', allow_pickle=True)

# pacifci polygon masks
final_polygon_mask_pacific = np.load('../data/polygon_data/Pacific_final_polygon_mask.npy', allow_pickle=True)
final_polygon_vertices_pacific = np.load('../data/polygon_data/Pacific_final_polygon_vertices.npy', allow_pickle=True)
cluster_centers_pacific = np.load('../data/polygon_data/Pacific_final_cluster_centers.npy', allow_pickle=True)

## Figure 3

In [ ]:
%%time
from scipy.ndimage import gaussian_filter

plt.rcParams.update({'font.size': 13})

fig = plt.figure(figsize=(14,12))
gs = gridspec.GridSpec(4, 3, width_ratios=[1.5, 1, 1])

sel_region = ['Atlantic', 'Atlantic', 'Pacific', 'Pacific']
sel_polygon = [136, 111, 64, 0]
surf_alk = [f'../data/plume_surf_excess_alk/North_{r}_basin-{p*4:04d}.nc' for r,p in zip(sel_region, sel_polygon)]

colors = ['blue', 'cyan', 'red', 'orange']
labels = ['Jan', 'Apr', 'Jul', 'Oct']
lon_mins = [-120, -120, 60, 60]
lon_maxs = [50, 50, 300, 300]
lat_mins = [0, -10, -50, -45]
lat_maxs = [90, 80, 80, 80]
central_longitudes = [0, 0, 180, 180]
region_names  = ['W. Subtrop. Atlantic', 'E. Subtrop. Atlantic', 'W. coast of US', 'W. coast of Colombia']
years_contour = [1, 2, 4, 8, 10]
linecolor_contour = cm.viridis(np.linspace(0, 1, 5))

def modify_ax_alk(ax):
    ax.set_ylim(5000, 1)
    ax.set_yscale('log')
    
    custom_y_ticks = [10, 100, 500, 1000, 4000]
    custom_y_labels  = [str(num) for num in custom_y_ticks]
    ax.set_yticks(custom_y_ticks)
    ax.set_yticklabels(custom_y_labels);
    
    ax.set_xlim(-0.01, 0.6)
    custom_x_ticks = np.arange(0, 0.7, 0.1)
    custom_x_labels  = [0, 0.1, 0, 0.1, 0, 0.1, 0.2]
    custom_x_labels = [str(num) for num in custom_x_labels]
    ax.set_xticks(custom_x_ticks)
    ax.set_xticklabels(custom_x_labels);


for i in range(4): # polygon
    
    for j in range(3):
        if j == 0:
            ax = plt.subplot(gs[i, j], projection=ccrs.PlateCarree(central_longitude=central_longitudes[i]))
            ax.text(-0.2, 0.5, f'{region_names[i]}', ha='center', va='center', rotation=90, transform=ax.transAxes, fontsize=14)  

            ### add map
            ds = xr.open_dataset(surf_alk[i])
            ds_ = util.pop_add_cyclic(ds)

            lon_min = lon_mins[i]
            lon_max = lon_maxs[i]
            lat_min = lat_mins[i]
            lat_max = lat_maxs[i]

            for y in range(len(years_contour)):
                
                c = ds_.surf_ALK_excess[years_contour[y]*12-1, :, :]
                if np.all(np.isnan(c)):
                    continue

                if y >= 3:
                    c = gaussian_filter(c, sigma=0.8) # smoother with large sigma
                    if i == 2:
                        c = gaussian_filter(c, sigma=1.3)
            
                # Use a threshold to define "plume boundary"
                threshold = np.nanmax(c) * 0.15

                # Plot contour at that threshold
                cs = ax.contour(ds_.TLONG, ds_.TLAT, c, levels=[threshold], linewidths=2, colors=linecolor_contour[y], alpha=0.8, transform=ccrs.PlateCarree(),)
            
            #### modify axes
            ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
            ax.set_xticks(np.arange(lon_min+20, lon_max, 60), crs=ccrs.PlateCarree())
            
            if i <=1 :
                ax.set_yticks([0,30,60], crs=ccrs.PlateCarree())
            else:
                ax.set_yticks([-30, 0,30,60], crs=ccrs.PlateCarree())
            
            lon_formatter = LongitudeFormatter(zero_direction_label=False)
            lat_formatter = LatitudeFormatter()
            ax.xaxis.set_major_formatter(lon_formatter)
            ax.yaxis.set_major_formatter(lat_formatter) 

            ax.add_feature(cfeature.LAND, edgecolor='white')
            ax.imshow(imread('./lightearth.jpg'),origin='upper', transform=ccrs.PlateCarree(), extent=[-180, 180, -90, 90])
            

            sca = ax.pcolormesh(ds_.TLONG, ds_.TLAT, eta_max.eta_max, transform=ccrs.PlateCarree(), vmin=0.75, vmax=0.95, cmap='bone_r')
        
            ## add polygons
            if i == 0:
                plot_polygons(ax, final_polygon_mask_atlantic, final_polygon_vertices_atlantic, cluster_centers_atlantic, sel_polygon[i])
                ax.text(-137, 90, 'a, i', fontsize=14, fontweight='bold')
            elif i == 1:
                plot_polygons(ax, final_polygon_mask_atlantic, final_polygon_vertices_atlantic, cluster_centers_atlantic, sel_polygon[i])
                ax.text(-137, 82, 'b, i', fontsize=14, fontweight='bold')
            elif i == 2:
                plot_polygons(ax, final_polygon_mask_pacific, final_polygon_vertices_pacific, cluster_centers_pacific, sel_polygon[i])
                ax.text(-140, 84, 'c, i', fontsize=14, fontweight='bold')
            elif i == 3:
                plot_polygons(ax, final_polygon_mask_pacific, final_polygon_vertices_pacific, cluster_centers_pacific, sel_polygon[i])
                ax.text(-140, 84, 'd, i', fontsize=14, fontweight='bold')

                
        ################ oae and dor eff
        elif j == 1:
            ax = plt.subplot(gs[i, j])
            ax.text(-4, 1.13, 'ii', fontsize=14, fontweight='bold')
            
            tarray = np.arange(0, 180, 1)/12
            
            ax.plot(tarray, no_nans(oae_curves.OAE_efficiency.sel(region=sel_region[i], polygon=sel_polygon[i], season='January').values), linewidth=2, color='b', label=rf'$\gamma_{{\mathrm{{OAE}}}}$')
            ax.plot(tarray, no_nans(dor_curves.DOR_efficiency.sel(region=sel_region[i], polygon=sel_polygon[i], season='January').values), linewidth=2, color='r', label=rf'$\gamma_{{\mathrm{{DOR}}}}$')
            ax.set_ylim(0, 1.1)
            ax.set_ylabel('Efficiency')
            ax.set_yticks(np.arange(0,1.1,0.2))

            
            if i == 3:
                ax.set_xlabel('Time since release (years)')
                custom_x_ticks = [0, 1, 2, 4, 8, 15]
                custom_x_labels  = [str(num) for num in custom_x_ticks]
                ax.set_xticks(custom_x_ticks)
                ax.set_xticklabels(custom_x_labels);
            else:
                ax.set_xticks([])

            # add vertical lines
            years_highlight = [1, 2, 4, 8]
            for p in range(len(years_highlight)):
                ax.axvline(years_highlight[p], linestyle='--', color='k',linewidth=1, )
                
            if i == 0:
                ax.legend(loc='lower right', fontsize=10)

        ################ ratios
        elif j == 2:
            ax = plt.subplot(gs[i, j])
            ax.text(-4, 0.846, 'iii', fontsize=14, fontweight='bold')
            
            tarray = np.arange(0, 180, 1)/12

            oae_eff = no_nans(oae_curves.OAE_efficiency.sel(region=sel_region[i], polygon=sel_polygon[i], season='January').values)
            dor_eff = no_nans(dor_curves.DOR_efficiency.sel(region=sel_region[i], polygon=sel_polygon[i], season='January').values)
            w_eta_max = no_nans(weighted_eta_max.weighted_eta_max.sel(region=sel_region[i], polygon=sel_polygon[i], season='January').values)
            
            ax.plot(tarray, oae_eff/dor_eff, linewidth=2, color='brown', label=rf'$\gamma_{{\mathrm{{OAE}}}}$/$\gamma_{{\mathrm{{DOR}}}}$')
            ax.plot(tarray, w_eta_max, linewidth=1.6, color='k', linestyle='solid', label=r'$\overline{\eta_{\mathrm{max}}}$')

            ax.set_ylim(0.785, 0.845)
            ax.set_ylabel('')
            if i == 3:
                ax.set_xlabel('Time since release (years)')
                custom_x_ticks = [0, 1, 2, 4, 8, 15]
                custom_x_labels  = [str(num) for num in custom_x_ticks]
                ax.set_xticks(custom_x_ticks)
                ax.set_xticklabels(custom_x_labels);
            else:
                ax.set_xticks([])

            # add vertical lines
            years_highlight = [1, 2, 4, 8]
            for p in range(len(years_highlight)):
                ax.axvline(years_highlight[p], linestyle='--', color='k',linewidth=1, )
                
            if i == 0:
                ax.legend(loc='upper left', fontsize=10)

# Create a new axis for the colorbar
cax = fig.add_axes([0.125, 0.07, 0.14, 0.01])  # [ left, bottom, width, height ]
#cax.set_aspect(0.5)  # Adjust width
#cax.set_aspect(0.5)  # Adjust height
cmap = cm.get_cmap('bone_r')  # Choose a colormap
normalize = plt.Normalize(vmin=0.75, vmax=0.95)  # Normalize the color values
sm = cm.ScalarMappable(cmap=cmap, norm=normalize)
cbar = fig.colorbar(sm, cax=cax, shrink=0.5, label=rf'$\eta_{{\mathrm{{max}}}}$', orientation='horizontal')
cbar.ax.tick_params(labelsize=12)
ticks = [0.75, 0.80, 0.85, 0.90, 0.95]
cbar.set_ticks(ticks)
cbar.set_ticklabels([str(t) for t in ticks])

ax_text = fig.add_axes([0.3, 0.07, 0.1, 0.01])
x_start = 0.3
x_spacing = 0.18 
ax_text.text(
        0.05, 0.5, f'Year',
        color='k', ha='center', va='center', fontsize=14,
    )
for i, (year, color) in enumerate(zip(years_contour, linecolor_contour)):
    if i == 4:
        add_space = ''
    else:
        add_space = ' '
    ax_text.text(
        x_start + i * x_spacing, 0.5, f' {year}{add_space}',
        color=color, ha='center', va='center', fontsize=14, 
    )

ax_text.axis('off')

plt.subplots_adjust(wspace=0.35, hspace=0.2)

plt.savefig('./figures/Figure_3.png', dpi=200, bbox_inches='tight')
plt.savefig('./figures/Figure_3.pdf', bbox_inches='tight')

## Figure S3

In [ ]:
grid = pop_tools.get_grid('POP_gx1v7')[['TAREA', 'KMT', 'TLAT', 'TLONG', 'REGION_MASK']]
tlong = grid.TLONG.values
tlat = grid.TLAT.values

# pacific
final_polygon_mask_pacific = np.load('../data/polygon_data/Pacific_final_polygon_mask.npy')
final_polygon_vertices_pacific = np.load('../data/polygon_data/Pacific_final_polygon_vertices.npy', allow_pickle=True)
cluster_centers_pacific = np.load('../data/polygon_data/Pacific_final_cluster_centers.npy', allow_pickle=True)

# atlantic
final_polygon_mask_atlantic = np.load('../data/polygon_data/Atlantic_final_polygon_mask.npy')
final_polygon_vertices_atlantic = np.load('../data/polygon_data/Atlantic_final_polygon_vertices.npy', allow_pickle=True)
cluster_centers_atlantic = np.load('../data/polygon_data/Atlantic_final_cluster_centers.npy', allow_pickle=True)

# south
final_polygon_mask_south = np.load('../data/polygon_data/South_final_polygon_mask_120EEZ_180openocean.npy')
final_polygon_vertices_south = np.load('../data/polygon_data/South_final_polygon_vertices_120EEZ_180openocean.npy', allow_pickle=True)
cluster_centers_south = np.load('../data/polygon_data/South_final_cluster_centers_120EEZ_180openocean.npy', allow_pickle=True)

# southern ocean
final_polygon_mask_SO = np.load('../data/polygon_data/Southern_Ocean_final_polygon_mask.npy')
final_polygon_vertices_SO = np.load('../data/polygon_data/Southern_Ocean_final_polygon_vertices.npy', allow_pickle=True)
cluster_centers_SO = np.load('../data/polygon_data/Southern_Ocean_final_cluster_centers.npy', allow_pickle=True)

all_region_masks = [final_polygon_mask_atlantic, final_polygon_mask_pacific, final_polygon_mask_south, final_polygon_mask_SO]
all_region_vertices = [final_polygon_vertices_atlantic, final_polygon_vertices_pacific, final_polygon_vertices_south, final_polygon_vertices_SO]
all_region_cluster_centers = [cluster_centers_atlantic, cluster_centers_pacific, cluster_centers_south, cluster_centers_SO]

In [ ]:
regs = ['North_Atlantic_basin', 'North_Pacific_basin', 'South', 'Southern_Ocean']
num_polys = [150, 200, 300, 40]
seasons = ['01', '04', '07', '10']

dict_regs = {
    'Atlantic': 'North_Atlantic_basin',
    'Pacific': 'North_Pacific_basin',
    'South': 'South',
    'Southern_Ocean': 'Southern_Ocean',
}

select = [
    'Alaska_Pacific_030_Jan.nc', 'Japan_Pacific_008_Jan.nc', 'Offshore Oman_South_242_Jan.nc', 'South Africa_South_011_Jan.nc',
    'Brazil_Atlantic_068_Jan.nc', 'Kerguelen_South_013_Jan.nc', 'Offshore W. Coast USA_Pacific_195_Jan.nc', 'S. India_South_015_Jan.nc',
    'E. Africa_South_050_Jan.nc', 'Madagascar_South_032_Jan.nc', 'Offshore W. Sahara_Atlantic_111_Jan.nc', 'S. USA_Atlantic_087_Jan.nc',
    'E. USA_Atlantic_017_Jan.nc', 'New Zealand_South_072_Jan.nc', 'Oman_South_087_Jan.nc', 'Tasmania_South_023_Jan.nc',
    'Hawaii_Pacific_007_Jan.nc', 'North Sea_Atlantic_016_Jan.nc', 'Patagonia_South_061_Jan.nc', 'W. Australia_South_045_Jan.nc',
    'Iceland_Atlantic_032_Jan.nc', 'Norway_Atlantic_099_Jan.nc', 'Peru_South_024_Jan.nc', 'W. Coast USA_Pacific_064_Jan.nc',
    'Indonesia_Pacific_020_Jan.nc', 'Offshore E. USA_Atlantic_134_Jan.nc', 'Portugal_Atlantic_011_Jan.nc', 'W. Sahara_Atlantic_027_Jan.nc',
    #'subpolar_Atlantic_100_Oct.nc', 'equator_Atlantic_046_Oct.nc', 
    'Labrador_Atlantic_000_Jan.nc', 
    #'labrador_Atlantic_000_Apr.nc', 'labrador_Atlantic_000_Jul.nc', 'labrador_Atlantic_000_Oct.nc', 
    'Equator_Pacific_000_Jan.nc',
]

extracted_info = []

for name in select:
    match = re.match(r'([^_]+(?:_[^_]+)*)_(Pacific|Atlantic|South)_(\d+)_([A-Za-z]+)\.nc$', name)
    if match:
        location, ocean, code, month = match.groups()
        extracted_info.append((location, ocean, code, month))

for case in extracted_info:
    print(case)

In [ ]:
def modify(ax):
    ax.set_extent([0, 360, -85, 80], crs=ccrs.PlateCarree())
    ax.set_xticks(np.arange(-180, 180, 60), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(-90, 90, 30), crs=ccrs.PlateCarree())
    lon_formatter = LongitudeFormatter(zero_direction_label=False)
    lat_formatter = LatitudeFormatter()
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter)  
    ax.imshow(imread('./lightearth.jpg'),origin='upper', transform=ccrs.PlateCarree(), extent=[-180, 180, -90, 90])

def plot_polygons(ax, mask, vertices, cluster_centers, p, write_polygon_index = True, poly_offset=0):
    # a list of colors
    colors = list(mcolors.TABLEAU_COLORS.values())
    ind_color = np.arange(len(colors)) # 0- 9

    # plot convex hull
    if len(vertices) >= 3:
        hull = ConvexHull(vertices) # get the indices of the periphery of a patch, for plotting purpose
        polygon = list(vertices[i] for i in hull.vertices.tolist() )
        polygon = np.array(polygon)
        ax.plot(np.append(polygon[:,0], polygon[0,0]), np.append(polygon[:,1], polygon[0,1]), 'k-', linewidth=0.5, transform=ccrs.PlateCarree())
        ax.fill(polygon[:, 0], polygon[:, 1], color='dimgray', alpha=0.5, transform=ccrs.PlateCarree())
    # plot polygon masks
    index = np.where(mask == 1)
    #ax.scatter(tlong[index], tlat[index], c=colors[random.choice(ind_color)], s=1, alpha=0.6, transform=ccrs.PlateCarree())
    if write_polygon_index:
        ax.text(cluster_centers[0]-2, cluster_centers[1]-1, str(p+poly_offset), fontsize=10, color='k', transform=ccrs.PlateCarree())
    
fig = plt.figure(figsize=(15,10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=208))

for case in extracted_info:
    
    r,p,s = case[1], int(case[2]), case[3]
    ind_reg = regs.index(dict_regs[r])     # region index. Atlantic is 0,....
    poly_offsets = [0,150, 150+200, 150+200+300]

    plot_polygons(ax, all_region_masks[ind_reg][p], all_region_vertices[ind_reg][p], all_region_cluster_centers[ind_reg][p], p, poly_offset=poly_offsets[ind_reg])


modify(ax)
fig.savefig('./figures/Figure_S3.png', dpi=200, bbox_inches='tight')

## Figure S4

In [ ]:
oae_curves = xr.open_dataset('../data/oae_efficiency_curves.nc')
dor_curves = xr.open_dataset('../data/dor_efficiency_curves.nc')
weighted_eta_max = xr.open_dataset('../data/weighted_eta_max_curves.nc') # weighted by excess alk
eta_max = xr.open_dataset('../data/eta_max_mean.nc')

In [ ]:
%%time
def plot_dor_oae_eff(loc, r, p, s, ax, fontsize=12):

    season_short_names = {'Jan': 'January', 'Apr': 'April', 'Jul': 'July', 'Oct':'October'}
    season_index = {'Jan': '01', 'Apr': '04', 'Jul': '07', 'Oct':'10'}
    
    #print(loc, r, p, s)

    ind_reg = regs.index(dict_regs[r])     # region index. Atlantic is 0,....
    poly_offsets = [0,150, 150+200, 150+200+300]

    ax.set_title(f'{loc} {p+poly_offsets[ind_reg]}', fontsize=fontsize)
    
    # dor curves
    dor_eff = dor_curves.sel(region=r, polygon=p).sel(season=season_short_names[s]).DOR_efficiency
    dor_eff = dor_eff[~np.isnan(dor_eff)]

    # corrected oae curves
    oae_eff = oae_curves.sel(region=r, polygon=p).sel(season=season_short_names[s]).OAE_efficiency
    oae_eff = oae_eff[~np.isnan(oae_eff)]

    # plot
    ax.plot(np.arange(0,180,1)/12, oae_eff, label=rf'$\gamma_{{\mathrm{{OAE}}}}$', color='b', linestyle='solid')
    ax.plot(np.arange(0,180,1)/12, dor_eff, label=rf'$\gamma_{{\mathrm{{DOR}}}}$', color='r', linestyle='solid')

    ax.set_ylim(0, 1.1)
            
fig, axs = plt.subplots(6, 5, figsize=(15, 12), sharex=True)
axs = axs.flatten()


for i in range(len(extracted_info)):
    case = extracted_info[i]
    r,p,s = case[1], int(case[2]), case[3]
    loc = case[0]
    plot_dor_oae_eff(loc, r,p,s, axs[i])
    

axs[0].legend()
for i in range(6):
    axs[-i].set_xlabel('Year')
for i in range(6):
    axs[i*5].set_ylabel('Efficiency')
plt.tight_layout()

fig.savefig('./figures/Figure_S4.png', dpi=200, bbox_inches='tight')

## Figure S5

In [ ]:
%%time
def plot_dor_oae_eff(loc, r, p, s, ax, fontsize=12):

    season_short_names = {'Jan': 'January', 'Apr': 'April', 'Jul': 'July', 'Oct':'October'}
    season_index = {'Jan': '01', 'Apr': '04', 'Jul': '07', 'Oct':'10'}

    #print(loc, r, p, s)
    ind_reg = regs.index(dict_regs[r])     # region index. Atlantic is 0,....
    poly_offsets = [0,150, 150+200, 150+200+300]

    ax.set_title(f'{loc} {p+poly_offsets[ind_reg]}', fontsize=fontsize)

    # dor curves
    dor_eff = dor_curves.sel(region=r, polygon=p, season=season_short_names[s]).DOR_efficiency
    dor_eff = dor_eff[~np.isnan(dor_eff)]

    # corrected oae curves
    oae_eff = oae_curves.sel(region=r, polygon=p, season=season_short_names[s]).OAE_efficiency
    oae_eff = oae_eff[~np.isnan(oae_eff)]

    # oae / dor 
    ax.plot(np.arange(0,180,1)/12, oae_eff/dor_eff, label=rf'$\gamma_{{\mathrm{{OAE}}}}$/$\gamma_{{\mathrm{{DOR}}}}$', color='brown', linestyle='solid')
    ax.set_ylim(0.78, 0.91)

    # weighted eta_max
    weighted_eta = weighted_eta_max.sel(region=r, polygon=p, season=season_short_names[s]).weighted_eta_max
    weighted_eta = weighted_eta[~np.isnan(weighted_eta)]
    
    ax.plot(np.arange(0,180,1)/12, weighted_eta, label=r'$\overline{\eta_{\mathrm{max}}}$', color='k', linestyle='solid')


fig, axs = plt.subplots(6, 5, figsize=(15, 12), sharex=True)
axs = axs.flatten()

i=0
for case in extracted_info:
    r,p,s = case[1], int(case[2]), case[3]
    loc = case[0]
    plot_dor_oae_eff(loc, r,p,s, axs[i])
    i += 1

axs[2].legend()
for i in range(6):
    axs[-i].set_xlabel('Year')

plt.tight_layout()

fig.savefig('./figures/Figure_S5.png', dpi=200, bbox_inches='tight')

## Figure 4

In [ ]:
whole_ds_oae = xr.open_dataset('../data/oae_eff_maps.nc')
whole_ds_dor = xr.open_dataset('../data/dor_eff_maps.nc')
whole_ds_weighted_eta_max = xr.open_dataset('../data/weighted_eta_max_maps.nc')

whole_ds_oae = util.pop_add_cyclic(whole_ds_oae)
whole_ds_dor = util.pop_add_cyclic(whole_ds_dor)
whole_ds_weighted_eta_max = util.pop_add_cyclic(whole_ds_weighted_eta_max)

In [ ]:
ratio = whole_ds_oae.OAE_efficiency.values / whole_ds_dor.DOR_efficiency.values
change_ratio = ratio - ratio[:,0,:,:][:, np.newaxis, :, :]
change_eta_max = (whole_ds_weighted_eta_max.weighted_eta_max - whole_ds_weighted_eta_max.weighted_eta_max.isel(N_month=0)).values

In [ ]:
change_ratio.shape

In [ ]:
change_ratio_mean = np.nanmean(change_ratio, axis=0)
change_eta_max_mean = np.nanmean(change_eta_max, axis=0)

In [ ]:
%%time
from scipy.stats import linregress
plt.rcParams.update({"font.size": 11})

FONTSIZE = 10
def modify(ax):
    ax.set_extent([0, 360, -85, 80], crs=ccrs.PlateCarree())
    lon_formatter = LongitudeFormatter(zero_direction_label=False)
    lat_formatter = LatitudeFormatter()
    ax.xaxis.set_major_formatter(lon_formatter)
    ax.yaxis.set_major_formatter(lat_formatter) 
    ax.imshow(imread('./lightearth.jpg'),origin='upper', transform=ccrs.PlateCarree(), extent=[-180, 180, -90, 90])

central_longitude=208

labels = ['a', 'b', 'c']

fig = plt.figure(figsize=(11,5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.5, 1.1], height_ratios=[1, 1], wspace=0.55)

# Separate the first column into 2 rows
gs_sub1 = gs[:, 0].subgridspec(2, 1, height_ratios=[1,  1], hspace=0.1)
ax = plt.subplot(gs_sub1[0, 0], projection=ccrs.PlateCarree(central_longitude=central_longitude))
ax1 = plt.subplot(gs_sub1[1, 0], projection=ccrs.PlateCarree(central_longitude=central_longitude))

# Separate the second column into 3 rows, and only use the middle one
gs_sub2 = gs[:, 1].subgridspec(3, 1, height_ratios=[1,3,1], hspace=0.1)
ax2 = plt.subplot(gs_sub2[1,0])


time_step = -1

############ Scatter
# Flatten arrays and mask NaNs
# y = change_ratio[:, time_step, :, :].flatten()
# x = change_eta_max[:, time_step, :, :].flatten()
y = change_ratio_mean[time_step, :, :].flatten()
x = change_eta_max_mean[time_step, :, :].flatten()
lat = whole_ds_dor.TLAT.values
#lat = np.broadcast_to(lat, (4, *lat.shape))
lat = lat.flatten()


# Remove NaNs
mask = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(lat)
x_clean = x[mask]
y_clean = y[mask]
lat_clean = lat[mask]

# Linear regression
slope, intercept, r_value, p_value, std_err = linregress(x_clean, y_clean)

# Plot
sc = ax2.scatter(x_clean, y_clean, c=lat_clean, s=5, vmin=-90, vmax=90, cmap='vanimo', alpha=0.6)

# Regression line
x_fit = np.linspace(np.min(x_clean), np.max(x_clean), 100)
y_fit = slope * x_fit + intercept
ax2.plot(x_fit, y_fit, 'k-', label=f'Fit: y = {slope:.2f}x + {intercept:.2f}\n$R^2$={r_value**2:.2f}, p<1e-5')

ax2.set_ylabel(rf'$\Delta(\gamma_{{\mathrm{{OAE}}}}$/$\gamma_{{\mathrm{{DOR}}}})$')
ax2.set_xlabel(r'$\Delta\overline{\eta_{\mathrm{max}}}$')
cbar = plt.colorbar(sc, ax=ax2, label='Latitude (°N)', shrink=0.8, ticks=[-60, -30, 0, 30, 60])
ax2.legend(fontsize=8.5)

ax2.set_xlim(-0.07, 0.07)
ax2.set_ylim(-0.07, 0.07)
ax2.set_xticks(np.arange(-0.06, 0.07, 0.03))
ax2.set_yticks(np.arange(-0.06, 0.07, 0.03))

########################## maps
vmin=-0.06
vmax=0.06
cmap='coolwarm'

ax.pcolormesh(whole_ds_dor.TLONG, whole_ds_dor.TLAT, change_eta_max_mean[time_step],
              transform=ccrs.PlateCarree(), cmap=cmap, vmin=vmin, vmax=vmax)
ax1.pcolormesh(whole_ds_dor.TLONG, whole_ds_dor.TLAT, change_ratio_mean[time_step],
              transform=ccrs.PlateCarree(), cmap=cmap, vmin=vmin, vmax=vmax)

ax.text(0.07, 0.87, labels[0], transform=fig.transFigure, fontsize=FONTSIZE+8)
ax1.text(0.07, 0.47, labels[1], transform=fig.transFigure, fontsize=FONTSIZE+8)
ax1.text(0.62, 0.75, labels[2], transform=fig.transFigure, fontsize=FONTSIZE+8)


ax.set_yticks([-60,-30,0,30,60], crs=ccrs.PlateCarree())
ax.set_yticklabels(ax.get_yticks(), fontsize=FONTSIZE)
ax1.set_xticks(np.arange(0, 360, 60), crs=ccrs.PlateCarree())
ax1.set_xticklabels(ax1.get_xticks(), fontsize=FONTSIZE)

ax1.set_yticks([-60,-30,0,30,60], crs=ccrs.PlateCarree())
ax1.set_yticklabels(ax1.get_yticks(), fontsize=FONTSIZE)
    
modify(ax)
modify(ax1)

ind_reg_sel = [0,0,1,1]
ind_polygon_sel = [136, 111, 64, 0]
for ind_reg, p in zip(ind_reg_sel, ind_polygon_sel):
    plot_polygons(ax, all_region_masks[ind_reg][p], all_region_vertices[ind_reg][p], all_region_cluster_centers[ind_reg][p], p, write_polygon_index=False)
    plot_polygons(ax1, all_region_masks[ind_reg][p], all_region_vertices[ind_reg][p], all_region_cluster_centers[ind_reg][p], p, write_polygon_index=False)

def add_colorbar(x0, y0, vmin, vmax, label, num_levels_ticks, cmap_label='viridis'):
    '''
    x0, y0: start location for the colorbar
    vmin, vmax: range of the colorbar
    label: label of the colorbar'
    '''
    cax = fig.add_axes([x0, y0, 0.01, 0.3])  # [x0, y0, width, height]
    cmap = plt.colormaps[cmap_label]
    normalize = plt.Normalize(vmin=vmin, vmax=vmax)  # Normalize the color values
    sm = cm.ScalarMappable(cmap=cmap, norm=normalize)
    cbar = fig.colorbar(sm, cax=cax, shrink=0.9, label=label, orientation='vertical', ticks=np.linspace(vmin, vmax, num_levels_ticks))
    cbar.ax.tick_params(labelsize=FONTSIZE)
    cbar.ax.xaxis.label.set_size(FONTSIZE)

add_colorbar(0.49, 0.55, vmin, vmax, r'$\Delta\overline{\eta_{\mathrm{max}}}$', 5, cmap_label=cmap)
add_colorbar(0.49, 0.15, vmin, vmax, rf'$\Delta(\gamma_{{\mathrm{{OAE}}}}$/$\gamma_{{\mathrm{{DOR}}}})$', 5, cmap_label=cmap)

plt.savefig('./figures/Figure_4.png', dpi=200, bbox_inches='tight')
plt.savefig('./figures/Figure_4.pdf', bbox_inches='tight')